
================================================================

**Author:** Carlos Stalin Saritama Atopo  
**Email:** cssaritama@gmail.com  
**Area of Specialization:** Analytics, Advanced Analytics, and Artificial Intelligence  
**Date:** August 2025 

===============================================================

# 03 — Modeling

**Goals**
- Train baseline and advanced models
- Hyperparameter tuning
- Calibrate decision thresholds
- Export predictions for backtesting

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report, f1_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

try:
    from xgboost import XGBClassifier
    xgb_ok = True
except Exception:
    xgb_ok = False

PROC_DIR = Path('../data/processed')
feat = pd.read_parquet(PROC_DIR / 'features.parquet')

exclude = {'date','ticker','open','high','low','close','volume','target'}
X = feat.drop(columns=[c for c in exclude if c in feat.columns])
y = feat['target']

cut1 = int(len(feat)*0.6); cut2 = int(len(feat)*0.8)
X_train, X_valid, X_test = X.iloc[:cut1], X.iloc[cut1:cut2], X.iloc[cut2:]
y_train, y_valid, y_test = y.iloc[:cut1], y.iloc[cut1:cut2], y.iloc[cut2:]

scaler = StandardScaler(with_mean=False)
X_train_s = scaler.fit_transform(X_train)
X_valid_s = scaler.transform(X_valid)
X_test_s  = scaler.transform(X_test)


## Train Baseline Models

In [ ]:
models = {}
models['DecisionTree'] = DecisionTreeClassifier(max_depth=6, random_state=42)
models['RandomForest'] = RandomForestClassifier(n_estimators=200, max_depth=8, n_jobs=-1, random_state=42)
models['Logistic'] = LogisticRegression(max_iter=200)
if xgb_ok:
    models['XGBoost'] = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0, random_state=42, tree_method='hist')

scores = {}
for name, clf in models.items():
    clf.fit(X_train_s, y_train)
    p = clf.predict_proba(X_valid_s)[:,1]
    auc = roc_auc_score(y_valid, p)
    scores[name] = auc

scores

## Select Best Model & Evaluate on Test

In [ ]:
best = max(scores, key=scores.get)
clf = models[best]

from scipy.sparse import vstack
X_trv = vstack([X_train_s, X_valid_s])
y_trv = pd.concat([y_train, y_valid])
clf.fit(X_trv, y_trv)

p_valid = clf.predict_proba(X_valid_s)[:,1]
ths = np.linspace(0.3,0.7,21)
fs = [f1_score(y_valid, (p_valid>t).astype(int)) for t in ths]
opt_t = ths[int(np.argmax(fs))]

p_test = clf.predict_proba(X_test_s)[:,1]
auc_test = roc_auc_score(y_test, p_test)
print('Best model:', best, ' Test ROC-AUC:', round(auc_test,4), ' Opt threshold:', round(opt_t,3))

## Export Predictions

In [ ]:
meta = feat.iloc[cut2:][['date','ticker']].reset_index(drop=True)
preds = pd.DataFrame({'date':meta['date'],'ticker':meta['ticker'],'pred_proba':p_test,'pred_label':(p_test>opt_t).astype(int)})

OUT_DIR = Path('../models')
OUT_DIR.mkdir(parents=True, exist_ok=True)
preds_path = OUT_DIR / 'predictions.csv'
preds.to_csv(preds_path, index=False)
preds_path, preds.head()